In [ ]:

# System
import time
import datetime
import os
from pathlib import Path
import random

# Torch
import torch
from torch import nn
from torch.cuda.amp import autocast
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset
from torchvision import transforms
from torchmetrics.classification import MulticlassRecall
from torchvision.ops import box_iou

# Clip
import clip

# Data manipulation
import pandas as pd
import numpy as np
import math

# Image manipulation
from PIL import Image
import cv2

In [ ]:
class colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    ORANGE = '\033[33m'
    RESULT =  '\033[94m'
    WARNING = '\033[93m'
    DEBUGGING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

def now():
    return datetime.datetime.now().strftime("%H:%M:%S")

def error(buffer):
    print(now()+" ["+colors.BOLD+colors.FAIL+"ERROR"+colors.ENDC+"] " + buffer +"\n")

def warning(buffer):
    print(now()+" ["+colors.BOLD+colors.ORANGE+"WARNING"+colors.ENDC+"] " + buffer +"\n")

def debugging(buffer):
    print(now()+" ["+colors.BOLD+colors.DEBUGGING+"DEBUG"+colors.ENDC+"] " + buffer +"\n")

def info(buffer):
    print(now()+" ["+colors.BOLD+colors.OKGREEN+"INFO"+colors.ENDC+"] " + buffer +"\n")
     


In [ ]:

get_device_first_call=True
def get_device():
    global get_device_first_call
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if get_device_first_call:
        info("The current device is " + device)
        get_device_first_call=False
    return device

def save_model(model, epoch, optimizer, total_loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': total_loss,
        }, path+"/personal_model_"+str(epoch)+".pt")

def load_model(model, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    return model, epoch, loss

class TensorBoard():
    # This class allows to log in tensorboard
    def __init__(self, log_dir):
        rootdir = log_dir
        max=1
        for file in os.listdir(rootdir):
            d = os.path.join(rootdir, file)
            if os.path.isdir(d) and file.startswith("exp"):
                num = int(file.replace("exp",""))
                if num > max:
                    max = num
        log_dir = log_dir+"/exp"+str(max+1)
        self.writer = SummaryWriter(log_dir=log_dir)

    def log_values(self, step, loss, accuracy, prefix):
        self.writer.add_scalar(f"{prefix}/loss", loss, step)
        self.writer.add_scalar(f"{prefix}/accuracy", accuracy, step)

    def close(self):
        self.writer.close()


The RefCOCO Dataset

The RefCOCO dataset is composed by about 50'000 images with at least an associated descriptions. We use the class RefCOCO to load, manage, order and elaborate both texts and descriptions. We are also able to augment data if requested, thanks to the class DataAugmentation that randomly generates some noise on the original image. In order to do everything, tha class performs these actions:

Load description and information of samples from the pickle file
Choose the type of splitted dataset (train, validation, test)
Set other parameters as batch, sample size ...
Create two lists which contain path to images and list of description ordered (description at index i refers to image at the same index i)
Encode data: load and preprocess images, tokenize texts and possibly augment data
The RefCOCO Class is implemented in a way that returns both preprocess data and normal data depending on which function is called.

During the Train-Validation-Test steps, it is needed to compute preprocess data, which means to have all the data already transformed to torch.tensor of size , which means that each RGB color gradient returns a matrix of . For this reason, the image data are transformed in tensor thanks to clip.preprocess function. Then, in the case of augment_data = True, which is crucial for the training split, the image data get through the DataAumentation class which performs the augmentation thanks to torchvision.tranforms tool. In this case, each training image is randomly augmented (this is due to memory size issues, having more augmentation yields better validation accuracy) between 5 different possibilities:

blur: this creates an image that adds blur to the original one;
rotate: transform the image by rotating it randomly up to 180°;
grayscale, transform the image into a grayscale one;
colorrand: randomly transform the color jitter of the image;
random_crop: randomly crop the image in a  tensor, to match the size of the original.
Then, as the data combine images and descriptions, each description is preprocessed through clip.tokenize, to retrieve a toch.tensor for each description. RefCOCOg has an average of 3.7 descriptions for each image, as having a different number of descriptions for every single image is unfeasible during the training process, and a fixed number is required, for each image that has at least one textual description, only one randomly chosen text is preprocessed and kept for the following steps.

Then the batch of preprocessed images and texts are retrieved by using the __get__item__ function during Train-Validation-Test, while to actual show the resutls in bounding bozes, as it is not required to have preprocessed items at the beginning, the images and texts are retrived with __getimg__ and __gettext__



In [ ]:
class DataAugmentation():
    # This class is used to perform tranformation in order to have augmented data

    def blur(img):
        # Gaussian Blur the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.GaussianBlur(kernel_size=5),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img

    def rotate(img):
        # Rotate the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomRotation(degrees=180),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img
    def grayscale(img):
        # Convert the image to grayscale
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img
    def colorrand(img):
        # Randomly change the color of the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ColorJitter(brightness=0.01*random.randrange(1,50), contrast=0.01*random.randrange(1,50), saturation=0.01*random.randrange(1,50), hue=0.01*random.randrange(1,50)),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img
    def random_crop(img):
        # Randomly crop the image
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomCrop((224,224)),
            transforms.ToTensor(),
        ])
        img = transform(img)
        return img

    def random_augmentation(img):
        # Randomly choose a transformation to be applied to the image
        n = random.randint(0, 5)
        if n == 0:
            return DataAugmentation.blur(img)
        elif n == 1:
            return DataAugmentation.rotate(img)
        elif n == 2:
            return DataAugmentation.grayscale(img)
        elif n == 3:
            return DataAugmentation.colorrand(img)
        else:
            return DataAugmentation.random_crop(img)


In [ ]:

class RefCOCO(Dataset):
    # This class is used to load the RefCOCO dataset, it is a subclass of Dataset.
    # It loads the dataset and to preprocess the images and the labels.
    # the requried parameters are:
    # annotations_file: the path to the pickle file containing the labels (RefCOCO folder provide a pickle file called refs(umd).p)
    # img_dir: the path to the folder containing the images (RefCOCO folder provide a subfolder called images)
    # model: the CLIP model
    # preprocess: the preprocessing function to be applied to the images
    # transform: the transformation to be applied to the images
    # target_transform: the transformation to be applied to the labels
    # device: the device to be used (cuda or cpu)
    # sample_size: the number of samples to be loaded in the dataset
    # batch_size: the batch size when __get_item__ is called
    # split_type: the split to be loaded (train, val, test)
    # augment_data: if True, the data is augmented with random transformations

    def __init__(self, annotations_file, img_dir, preprocess, device, sample_size, split_type, augment_data=False):
        x = pd.read_pickle(annotations_file)
        self.img_texts = pd.DataFrame(x)
        self.img_texts = self.img_texts.loc[self.img_texts['split'] == split_type]
        self.device = device
        self.sample_size = sample_size
        self.img_dir = img_dir
        self.preprocess = preprocess
        self.img, self.description = self.get_imgs_texts()
        info(split_type.upper()+": ENCODING"+(" & AUGMENTIG DATA..." if augment_data else "..."))
        self.enc_imgs, self.enc_txts = self.encode_data(augment_data)

    def __len__(self):
        return len(self.enc_imgs)

    def get_imgs_texts(self):
        # This function is used to get the images and the labels (descriptions) from the dataset
        # It returns a list of images and a list of list of sentences ordered (idx image = idx text)
        images=[]
        texts=[]
        index=0
        for _, el in self.img_texts.iterrows():
            images.append(self.img_dir+"/"+el["file_name"][0:27]+".jpg")
            sentences=[]
            for sent in el["sentences"]:
                sentences.append(sent["raw"])
            texts.append(sentences)
            index+=1
            if index >= self.sample_size:
                break
        return images, texts

    def encode_data(self, augment_data):
        # This function is used to encode data. It executes:
        # - Tokenization for text
        # - Preprocessing for images
        # - Randomly data augmentation if requested (1 original img = 1 noisy img)
        # the required parameter is:
        # augment_data: boolean that enable data augmentation
        enc_imgs=[]
        for img in self.img:
            enc_imgs.append(self.preprocess(Image.open(img)))
            if augment_data:
                tmp = self.preprocess(Image.open(img))
                enc_imgs.append(DataAugmentation.random_augmentation(tmp))

        enc_txts=[]
        for txt in self.description:
            enc_txts.append(clip.tokenize(txt[random.randint(0, len(txt)-1)]).to(self.device))
            if augment_data:
                enc_txts.append(clip.tokenize(txt[random.randint(0, len(txt)-1)]).to(self.device))

        return enc_imgs, enc_txts


    def __getimg__(self, idx):
        # This function is used to get image path and image name at index idx
        # Used only for evaluation purposes
        # the required parameter is:
        # idx: the index of the item to be returnedå
        image = self.img[idx]
        str_image = str(image)
        return image, str_image

    def __gettext__(self, idx):
        # This function is used to get list of descriptions at index idx
        # Used only for evaluation purposes
        # the required parameter is:
        # idx: the index of the item to be returned
        text = self.description[idx]
        return text

    def __getitem__(self, idx):
        # This function is used to get the item at the index idx
        # the required parameter is:
        # idx: the index of the item to be returned
        image = self.enc_imgs[idx]
        text = self.enc_txts[idx]

        return image, text


The following snippet has only the purpose to check if all the imports and the data load have been executed correctly, it has no functionalities inside the main program

In [ ]:
# RefCOCO requires (self, annotations_file, img_dir, preprocess, device, sample_size, split_type, augment_data=False)
annotations_file = 'dataset/refcocog/annotations/refs(umd).p'
root_imgs = 'dataset/refcocog/images'
_, preprocess = clip.load("RN50")
device = get_device()
check_loading = RefCOCO(annotations_file=annotations_file, img_dir = root_imgs, preprocess = preprocess, device = device, sample_size = 10, split_type = 'train', augment_data = True)
info("Data loaded in "+str(check_loading))



To load data into splitted Datasets with appropriate parameters and create DataLoaders for training, validation and test we implement the function get_data(...)


In [ ]:
def get_data(batch_size, annotations_file, img_root, test_batch_size = 16, preprocess = None, device = get_device(), sample_size_train = 42226, sample_size_test = 5023, sample_size_val = 2573, augment_data_train=True):
    # This function returns the train, test and eval data loaders
    # The data loaders are used to iterate over the data in batches
    # then they are used in the training, validatiing and testing loops
    # and are created using the RefCOCO class
    # the parameters are:
    # batch_size: the size of the batch used in the training loop
    # annotations_file: the path to the annotations file
    # img_root: the path to the images folder
    # test_batch_size: the size of the batch used in the testing and validation loops
    # preprocess: the preprocess function used to preprocess the images
    # device: the device used to run the code
    # sample_size_train: the number of samples to use in the training set
    # sample_size_test: the number of samples to use in the test set
    # sample_size_val: the number of samples to use in the validation set
    # augment_data_train: whether to augment the training data or not
    sample_size_train = sample_size_train if sample_size_train <= 42226 else 42226
    sample_size_test = sample_size_test if sample_size_test <= 5023 else 5023
    sample_size_val = sample_size_val if sample_size_val <= 2573 else 2573
    training_data = RefCOCO(annotations_file=annotations_file, img_dir=img_root, preprocess=preprocess, split_type='train', device=device, sample_size=sample_size_train, augment_data=augment_data_train)
    test_data = RefCOCO(annotations_file=annotations_file, img_dir=img_root, preprocess=preprocess, split_type='test', device=device, sample_size=sample_size_test)
    eval_data = RefCOCO(annotations_file=annotations_file, img_dir=img_root, preprocess=preprocess, split_type='val', device=device, sample_size=sample_size_val)

    num_training_samples = len(training_data)
    info("Number of training samples:" + str(num_training_samples))
    num_test_samples = len(test_data)
    info("Number of test samples:" + str(num_test_samples))
    num_eval_samples = len(eval_data)
    info("Number of eval samples:" + str(num_eval_samples))
    train_loader = torch.utils.data.DataLoader(training_data, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=test_batch_size, shuffle=False)
    eval_loader = torch.utils.data.DataLoader(eval_data, batch_size=test_batch_size, shuffle=False)
    return train_loader, test_loader, eval_loader
     
The Custom Clip Model MAIK

Library CustomClipMAIK (from MAnuel and erIK) is composed by 2 classes that extend torch.nn.Module.

The first BatchNorm1d is used for batch normalization for both batches texts and images
The second CustomClip allows us to define a new model, based on CLIP model, with our own weights, personal forward with bottlenecks for images and texts and a method to retrieve bounding boxes

class BatchNorm1d(torch.nn.Module):
  def __init__(self, in_features, track_running_stats=True, affine=True, momentum=0.9, device = 'cuda:0'):
    super().__init__()

    self.in_features = in_features
    self.track_running_stats = track_running_stats
    self.affine = affine

    self.device = device
    self.momentum = momentum
    if self.affine:
      self.gamma = torch.nn.Parameter(torch.ones(self.in_features, 1))
      self.beta = torch.nn.Parameter(torch.zeros(self.in_features, 1))

    if self.track_running_stats:
      # register_buffer registers a tensor as a buffer that will be saved as part of the model
      # but which does not require to be trained, differently from nn.Parameter
      self.register_buffer('running_mean', torch.zeros(self.in_features, 1))
      self.register_buffer('running_std', torch.ones(self.in_features, 1))

  def forward(self, x):
    # transpose (N, C) to (C, N)
    x = x.to(self.device)
    x = x.transpose(0, 1).contiguous().view(x.shape[1], -1).to(self.device)

    # calculate batch mean
    mean = x.mean(dim=1).view(-1, 1).to(self.device)

    # calculate batch std
    std = x.std(dim=1).view(-1, 1).to(self.device)

    # during training keep running statistics (moving average of mean and std)
    if self.training and self.track_running_stats:
      # no computational graph is necessary to be built for this computation
      with torch.no_grad():
        self.running_mean = self.running_mean.to(self.device)
        self.running_std = self.running_std.to(self.device)
        self.running_mean = (self.momentum * self.running_mean + (1 - self.momentum) * mean)
        self.running_std = (self.momentum * self.running_std + (1 - self.momentum) * std)

    # during inference time
    if not self.training and self.track_running_stats:
      mean = self.running_mean
      std = self.running_std

    # normalize the input activations
    x = x.to(self.device)
    mean = mean.to(self.device)
    std = std.to(self.device)
    x = (x - mean) / std

    # scale and shift the normalized activations
    if self.affine:
      x = x * self.gamma.to(self.device) + self.beta.to(self.device)

    return x.transpose(0, 1)
     
CustomClip

CustomClip inherit the toch.nn.Module to add layers and properties to various types of Neural Network. In this particular case, this class is used to provide useful fine-tuning methods of the original Clip-Model, and also some addition to providing bounding boxes out of the main algorithm to accomplish the Visual Grounding task. There are 4 main core parts in this class:

set_img_bottlneck which adds 3 linear layers activated by a Relu function, this update the weights by dividing the number of input features by 2 in each layer and then returning the same numbers of output features as in the beginning.
set_txt_bottleneck: it is similar to the images' one, but instead of the Relu activation function, it is activated by a Sigmoid, which yields better results in case of text classification as it tends to return values of .
__get_boxes__: this method crops the image based on the detector model (Yolov5s), and then passes it through the clip model to obtain a prediction of the descriptions in bounding boxes.
forward: this is the main function of the Model as it's the one that gets the inputs, does all the evaluations, and then returns the results. It is related to the Clip's one, as it encodes both text and image and then rescales them in order to maximize predictions, yet there the additions of the 2 aforementioned bottlenecks to enhance performance, and then batch normalization is applied both to text' and images' tensors
Architecture

ARCHITECTURE.jpg


class CustomClip(torch.nn.Module):
    # This class allows us to build an ad-hoc class based on CLIP, so we can edit the forward step
    # by introducing some techniques as batch normalization, bottleneck, ...
    # The class is based on the CLIP class, so we can use the same methods and attributes
    # Moreover we add a function to predict bounding boxes for the final goal

    def __init__(self, device, batch_size=128, norm=True, bias=True):

        super().__init__()
        self.device = device
        self.model, self.preprocess = clip.load('RN50', device=self.device, jit=False)
        self.detector = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True, _verbose=False)
        self.in_features = 1024
        self.out_features = 1024
        self.norm = norm
        if self.norm:
          self.bn1 = BatchNorm1d(self.in_features, track_running_stats=False, affine=False, momentum=0.5, device=self.device)
        self.bias = False
        self.norm = norm
        self.batch_size = batch_size
        self.img_bottleneck = self.set_img_bottleneck()
        self.txt_bottleneck = self.set_txt_bottleneck()
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def set_img_bottleneck(self):
        # Implement a bottleneck for images
        layer = [
                    torch.nn.Linear(self.in_features, self.in_features // 2, bias=self.bias),
                    torch.nn.ReLU(inplace=True),
                    torch.nn.Linear(self.in_features // 2, self.in_features // 2, bias=self.bias),
                    torch.nn.ReLU(inplace=True),
                    torch.nn.Linear(self.in_features // 2, self.out_features, bias=self.bias),
                ]
        bottleneck = torch.nn.Sequential(*layer).to(self.device)
        return bottleneck

    def set_txt_bottleneck(self):
       # Implement a bottleneck for texts
       layer = [
                    torch.nn.Linear(self.in_features, self.in_features // 2, bias=self.bias),
                    torch.nn.Sigmoid(),
                    torch.nn.Linear(self.in_features // 2, self.in_features // 2, bias=self.bias),
                    torch.nn.Sigmoid(),
                    torch.nn.Linear(self.in_features // 2, self.out_features, bias=self.bias),
       ]
       bottleneck = torch.nn.Sequential(*layer).to(self.device)
       return bottleneck

    def __get_model__(self):
        # This function returns the model and the preprocess function
        return self.model, self.preprocess

    def __get_boxes__(self, input_img, input_text):
        # This function returns the bounding box for the input image that fits more with the textual descritpion

        self.eval()

        # First we extract the bounding boxes from the image
        detections = self.detector(input_img).pandas().xyxy[0]
        image = Image.open(input_img)
        img_cropped = []
        bounding_boxes = []
        for _, item in detections.iterrows():
          xmin = int(item["xmin"])-30 if int(item["xmin"])-30 >= 0 else 0
          ymin = int(item["ymin"])-30 if int(item["ymin"])-30 >= 0 else 0
          xmax = int(item["xmax"])+30 if int(item["xmax"])+30 <= image.size[0] else image.size[0]
          ymax = int(item["ymax"])+30 if int(item["ymax"])+30 <= image.size[1] else image.size[1]
          cropped = image.crop((xmin, ymin, xmax, ymax))
          img_cropped.append(cropped)
          bounding_boxes.append({"xmin": int(item["xmin"]), "ymin": int(item["ymin"]), "xmax": int(item["xmax"]), "ymax": int(item["ymax"])})

        # If there are no bounding boxes, we return None
        if(len(img_cropped)==0):
           return None

        # Then we extract the features from the image and the text
        with torch.no_grad():
          tokenized_text = clip.tokenize([input_text]).squeeze(1).to(self.device)

          preprocessed_imgs=[]
          for index, img in enumerate(img_cropped):
            preprocessed_imgs.append(self.preprocess(img).to(self.device))
          preprocessed_imgs = torch.stack(preprocessed_imgs)

          # Pass text and cropped images to the model and return that one with highest score
          self.float()
          _, logits_per_text = self(preprocessed_imgs, tokenized_text)
          probs = logits_per_text.softmax(dim=1)
          top_prob, top_label = probs.topk(1, dim=-1)

        return bounding_boxes[top_label.item()], top_prob.item()

    def forward(self, image, text):
        # This function is the forward step of the model

        # First we extract the features from the image and the text and pass them into the bottleneck
        image = self.model.encode_image(image)
        with autocast(dtype=torch.half):
           image = self.img_bottleneck(image).to(self.device)
        text = self.model.encode_text(text)
        with autocast(dtype=torch.half):
           text = self.txt_bottleneck(text).to(self.device)

        # Then we normalize the features and compute the logits
        if self.norm:
            image = self.bn1(image).to(self.device)
            text = self.bn1(text).to(self.device)

        image = image / image.norm(dim=-1, keepdim=True).float()
        text = text / text.norm(dim=-1, keepdim=True).float()
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image @ text.t()
        logits_per_text = logits_per_image.t()
        return logits_per_image, logits_per_text
     
The Training & Test
Optimizer and Cost Function

def get_cost_function():
    # This function returns the cost function
    # the CrossEntropyLoss() is the most suitable for this task
    # because it is a classification task
    return torch.nn.CrossEntropyLoss()

def get_optimizer(net, lr, wd):
    # This function returns the optimizer
    # the Adam optimizer is the most suitable for this task
    # the parameters are the parameters of the network, the learning rate and the weight decay
    # net: the network, model
    # lr: learning rate
    # wd: weight decay
    return torch.optim.Adadelta(net.parameters(), lr=lr, weight_decay = wd)

def update_parameters(learning_rate, weight_decay, alpha):
    # This function updates the parameters of the optimizer
    # thorugh the Metropolis criterion of the simulated annealing
    # learning_rate: the learning rate of the optimizer
    # weight_decay: the weight decay of the optimizer
    # alpha: the alpha of the simulated annealing

    learning_rate = learning_rate * alpha
    weight_decay = weight_decay * alpha
    alpha = alpha/(alpha+0.001)
    return learning_rate, weight_decay, alpha


     
The above functions set the main parameters and methods required to perform the Training-Test process. Particularly, here are set the loss function and the optimizer.

CrossEntropyLoss: the model uses the Cross Entropy as loss function to retrieve the errors of the model, and then backpropagate it to activate the learning mechanism. Corss Entropy is the most suited one as it is a multiclass classification task, a also by doing some test, it is the function that returned by fare the best accuracy score.
Adadelta: the optimization function that was decided to use is Adadelta, which is a less aggressive implemnetation of Adagrad, and due to the complexity of finding the optimum in this cost functions it was necessary a smoother optimizer. In the following images it is represent how it returned far better accuracy score than the most common one, Stochastic Gradient Descent, during some test.
Screenshot from 2023-06-12 11-19-32.png

Image: Training and Validation using SGD optimizer(Blue) and Adadelta optimizer (Pink)

image.png

Final model with optimizer Adadelta, 60K images (30K come from data augmention), batch size of 16 and bottlenecks ReLU for image and Sigmoid for texts

Parameters

These are the parameters required to start the training sessions. It is important to note that this Colab Machine is only for demonstration purposes, and it doesn't provide a machine with enough hardware capacity to perform the actual training parameters to return a good-trained model. Due to this reason, the parameters are much lower than they should be. After the demonstrations, the model trained with a high amount of data is going to be loaded from Google Drive.


#DATASET PARAMS
#sample_size_train=30000
#sample_size_test=5000
#sample_size_val=516
sample_size_train=2000
sample_size_test=500
sample_size_val=500
augment_data_train=True

#TRAINING PARAMS
batch_size = 16
test_batch_size = 16
epochs = 10

#OPTIMIZER & LOSS PARAMS
cost_function = get_cost_function()
learning_rate = 0.0015
weight_decay = 0.000001
alpha = 1 # to decrease lr over time
     
The steps

Here, we implemented the step for the training (with backpropagation) and for the test (without it)


def training_step(model, train_dataloader,  optimizer, cost_function=get_cost_function(), device=get_device()):
    cumulative_accuracy = 0.0
    cumulative_loss = 0.0
    samples = 0.0
    clip.model.convert_weights(model)
    model.train()
    for (images, texts) in train_dataloader:
        # Zero the grad
        optimizer.zero_grad()
        # Prepare images and texts
        images = images.to(device)
        texts = texts.squeeze(1).to(device)
        # Build the ground truth
        ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
        # Compute the predictions
        logits_per_image, logits_per_texts = model(images, texts)
        # Calculate loss & backward it
        img_loss = cost_function(logits_per_image, ground_truth)
        desc_loss = cost_function(logits_per_texts, ground_truth)
        loss = (img_loss + desc_loss)/2
        loss.backward()
        optimizer.step()
        # Keep track of loss and accuracy metrics to see epochs progress
        cumulative_loss += loss.item()
        samples += images.shape[0]
        _, predicted = logits_per_image.max(dim=1)
        cumulative_accuracy += predicted.eq(ground_truth).sum().item()
        clip.model.convert_weights(model)
    return cumulative_loss / samples, cumulative_accuracy / samples


def test_step(model, test_loader, cost_function, device=get_device()):
    samples = 0.0
    cumulative_loss = 0.0
    cumulative_accuracy = 0.0
    model.eval()
    # disable gradient computation (we are only testing, we do not want our model to be modified in this step!)
    with torch.no_grad():
        # iterate over the test set
        for (images, texts) in test_loader:
            images = images.to(device)
            texts = texts.squeeze(1).to(device)
            logits_per_image, logits_per_texts = model(images, texts)
            ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
            img_loss = cost_function(logits_per_image, ground_truth)
            desc_loss = cost_function(logits_per_texts, ground_truth)
            loss = (img_loss + desc_loss)/2
            cumulative_loss += loss.item()
            samples += images.shape[0]
            _, predicted = logits_per_image.max(dim=1)
            cumulative_accuracy += predicted.eq(ground_truth).sum().item()

    return cumulative_loss / samples, cumulative_accuracy / samples
     
Let's start the training!

Finally we can put all parts together: load dataset, instanciate the custom clip, test accuracy, log epochs and start training.

NOTE: As explained earlier, the next Training-Test epoch are only to show the functionality of the Network, it doesnt' have enough data to return good results.

First, it is necessary to initialize the data loader required to perform the next steps


annotations_file = 'dataset/refcocog/annotations/refs(umd).p'
root_imgs = 'dataset/refcocog/images'
clip_model = CustomClip(device=get_device(), batch_size=batch_size, norm=True, bias=False)
#clip_model.float()
_ , clip_processor = clip_model.__get_model__()
optimizer = get_optimizer(clip_model, learning_rate, weight_decay)

train_loader, test_loader, val_loader = get_data(batch_size, annotations_file=annotations_file, img_root=root_imgs, test_batch_size = test_batch_size, preprocess=clip_processor, sample_size_train=sample_size_train, sample_size_test=sample_size_test, sample_size_val=sample_size_val, augment_data_train=augment_data_train)

     
Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
08:27:01 [INFO] TRAIN: ENCODING & AUGMENTIG DATA...

08:28:10 [INFO] TEST: ENCODING...

08:28:20 [INFO] VAL: ENCODING...

08:28:27 [INFO] Number of training samples:4000

08:28:27 [INFO] Number of test samples:500

08:28:27 [INFO] Number of eval samples:500


tb = TensorBoard("run")

info("BEFORE TRAINING...")
loss, accuracy = test_step(clip_model, train_loader, cost_function)
info("Train - LOSS: {:.4} ACCURACY: {:2.1%}".format(loss, accuracy))
tb.log_values(0, loss, accuracy, "Train")
loss, accuracy = test_step(clip_model, val_loader, cost_function)
info("Validation - LOSS: {:.4} ACCURACY: {:2.1%}".format(loss, accuracy))
tb.log_values(0, loss, accuracy, "Validation")
loss, accuracy = test_step(clip_model, test_loader, cost_function)
info("Test - LOSS: {:.4} ACCURACY: {:2.1%}".format(loss, accuracy))
tb.log_values(0, loss, accuracy, "Test")
optimizer = get_optimizer(clip_model, learning_rate, weight_decay)

     
08:28:33 [INFO] BEFORE TRAINING...

08:28:44 [INFO] Train - LOSS: 0.1793 ACCURACY: 5.8%

08:28:47 [INFO] Validation - LOSS: 0.1783 ACCURACY: 9.4%

08:28:48 [INFO] Test - LOSS: 0.1771 ACCURACY: 5.6%


info("INIT TRAINING...")
for ep in range(1, epochs+1):
    info("EPOCH "+str(ep)+":")
    loss, accuracy = training_step(clip_model, train_loader, optimizer, cost_function)
    save_model(clip_model, ep, optimizer, loss, "Personal_Model")
    info("Train - LOSS: {:.4} ACCURACY: {:2.1%} ".format(loss, accuracy))
    tb.log_values(ep, loss, accuracy, "Train")
    loss, accuracy = test_step(clip_model, val_loader, cost_function)
    info("Validation - LOSS: {:.4} ACCURACY: {:2.1%}".format(loss, accuracy))
    tb.log_values(ep, loss, accuracy, "Validation")
    learning_rate, weight_decay, alpha = update_parameters(learning_rate, weight_decay, alpha)
    optimizer = get_optimizer(clip_model, learning_rate, weight_decay)
     
08:28:48 [INFO] INIT TRAINING...

08:28:48 [INFO] EPOCH 1:

08:30:48 [INFO] Train - LOSS: 0.1501 ACCURACY: 20.9% 

08:30:50 [INFO] Validation - LOSS: 0.1133 ACCURACY: 41.4%

08:30:50 [INFO] EPOCH 2:

08:33:07 [INFO] Train - LOSS: 0.1112 ACCURACY: 43.7% 

08:33:09 [INFO] Validation - LOSS: 0.09591 ACCURACY: 49.4%

08:33:09 [INFO] EPOCH 3:

08:35:34 [INFO] Train - LOSS: 0.09059 ACCURACY: 55.5% 

08:35:36 [INFO] Validation - LOSS: 0.08437 ACCURACY: 56.0%

08:35:36 [INFO] EPOCH 4:

08:38:02 [INFO] Train - LOSS: 0.07578 ACCURACY: 63.2% 

08:38:04 [INFO] Validation - LOSS: 0.07748 ACCURACY: 59.0%

08:38:04 [INFO] EPOCH 5:

08:40:24 [INFO] Train - LOSS: 0.06257 ACCURACY: 72.4% 

08:40:26 [INFO] Validation - LOSS: 0.07429 ACCURACY: 62.8%

08:40:26 [INFO] EPOCH 6:

08:42:56 [INFO] Train - LOSS: 0.05307 ACCURACY: 77.5% 

08:42:57 [INFO] Validation - LOSS: 0.07146 ACCURACY: 64.6%

08:42:57 [INFO] EPOCH 7:

08:45:22 [INFO] Train - LOSS: 0.04457 ACCURACY: 83.2% 

08:45:23 [INFO] Validation - LOSS: 0.06826 ACCURACY: 64.8%

08:45:23 [INFO] EPOCH 8:

08:47:55 [INFO] Train - LOSS: 0.03709 ACCURACY: 86.2% 

08:47:57 [INFO] Validation - LOSS: 0.06738 ACCURACY: 66.4%

08:47:57 [INFO] EPOCH 9:

08:50:26 [INFO] Train - LOSS: 0.03204 ACCURACY: 88.9% 

08:50:28 [INFO] Validation - LOSS: 0.06611 ACCURACY: 67.0%

08:50:28 [INFO] EPOCH 10:

08:52:56 [INFO] Train - LOSS: 0.02699 ACCURACY: 91.4% 

08:52:58 [INFO] Validation - LOSS: 0.06432 ACCURACY: 68.0%


info("AFTER TRAINING...")
loss, accuracy = test_step(clip_model, train_loader, cost_function)
info("Train - LOSS: {:.4} ACCURACY: {:2.1%}".format(loss, accuracy))
tb.log_values(epochs+1, loss, accuracy, "Train")
loss, accuracy = test_step(clip_model, val_loader, cost_function)
info("Validation - LOSS: {:.4} ACCURACY: {:2.1%}".format(loss, accuracy))
tb.log_values(epochs+1, loss, accuracy, "Validation")
loss, accuracy = test_step(clip_model, test_loader, cost_function)
info("Test - LOSS: {:.4} ACCURACY: {:2.1%}".format(loss, accuracy))
tb.log_values(epochs+1, loss, accuracy, "Test")
tb.close()
     
08:52:58 [INFO] AFTER TRAINING...

08:53:09 [INFO] Train - LOSS: 0.01964 ACCURACY: 95.9%

08:53:11 [INFO] Validation - LOSS: 0.06432 ACCURACY: 68.0%

08:53:12 [INFO] Test - LOSS: 0.06589 ACCURACY: 65.4%


%load_ext tensorboard
%tensorboard --logdir=run
     
Evaluation & Results
For the evaluation part we mount our model saved during train (at the best epoch, to avoid under or over fitting). We chose 2 different type of metrics:

Recall: to measure the grounding accuracy
Cosine similarity: to measure semantic similarity
We implemented also an Intersection over Union metric, but we won't report here: it's able to measure the localization precision, but since it comes from Yolo architecture it isn't upgradable thanks to our network, so it's the same reported in various papers of ultralytics; moreover, there isn't any ground truth to compare with both in evaluation and training phase, so we are not able to train the network for this (unless labeling images one by one).


def recall(preds, target, num_labels, device = get_device()):
    # Recall is the fraction of relevant instances that have been retrieved over the total amount of relevant instances.
    # In the case of multiclass classification, the recall is the average of the recall of each class.
    # In the case of multilabel classification, the recall is the average of the recall of each label.
    # Parameters:
    # preds (torch.Tensor): Predicted values
    # target (torch.Tensor): Ground truth values
    # num_labels (int): Number of labels
    recall = MulticlassRecall(num_classes=num_labels, average=None).to(device)
    multilabel_recall = recall(preds, target)
    return multilabel_recall

def intersection_over_union(preds, target):
    # Intersection over union is a measure of the overlap between two bounding boxes.
    # It is calculated by dividing the area of overlap by the area of union.
    # Parameters:
    # preds (dictionary contains xmin, xmax, ymin, ymax): Predicted values
    # target (dictionary contains xmin, xmax, ymin, ymax): Ground truth values
    preds = torch.Tensor([preds["xmin"], preds["ymin"], preds["xmax"], preds["ymax"]])
    target = torch.Tensor([target["xmin"], target["ymin"], target["xmax"], target["ymax"]])
    value = box_iou(preds, target)
    return value.item()

def cosine_similarity(logits_img, logits_txt):
    # Semantic similarity is a metric defined over a set of documents or terms,
    # where the idea of distance between them is based on the likeness of their meaning or semantic content
    # Parameters:
    # custom_model (torch.nn.Module): custom model
    # img (string): Image
    # text (string): Description
    cos_sim = torch.nn.CosineSimilarity()
    similarity = cos_sim(logits_img, logits_txt)
    return similarity

def eval_step(model, eval_loader, device = get_device()):
    # Same as test_step, but with recall and cosine similarity
    samples = 0.0
    cumulative_accuracy = 0.0
    cumulative_recall = 0.0
    cumulative_sim = 0.0
    cumulative_accuracy = 0.0
    model.eval()
    model.float()
    # disable gradient computation (we are only testing, we do not want our model to be modified in this step!)
    with torch.no_grad():
        # iterate over the set
        for (images, texts) in eval_loader:
            images = images.to(device)
            texts = texts.squeeze(1).to(device)
            logits_per_image, logits_per_texts = model(images, texts)
            ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
            samples += images.shape[0]
            n_labels = logits_per_texts.shape[1]
            _, predicted = logits_per_image.max(dim=1)
            cumulative_recall += torch.sum(recall(predicted, ground_truth, n_labels, device)).item()
            cumulative_accuracy += predicted.eq(ground_truth).sum().item()
            cumulative_sim += torch.sum(cosine_similarity(logits_per_image, logits_per_texts)).item()

    return cumulative_accuracy / samples, cumulative_recall / samples, cumulative_sim / samples
     

annotations_file = 'dataset/refcocog/annotations/refs(umd).p'
root_imgs = 'dataset/refcocog/images'
model_path = 'model/FINAL_MODEL_personal_model_Adadelta_60K-16-aug-sigmoid.pt'
clip_model_ = CustomClip(device=get_device())
_, preprocess = clip_model_.__get_model__()
clip_model, epoch, loss = load_model(clip_model_, model_path)
test_data = RefCOCO(annotations_file = annotations_file, img_dir=root_imgs, preprocess = preprocess, split_type='test', device=get_device(), sample_size=1000)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=16, shuffle=True)

# Evaluate recall (grounding accuracy metric) and cosine similarity (semantic similarity metric)
info("EVALUATING...")
acc, rec, sim = eval_step(clip_model, test_loader)
angle = math.degrees(math.acos(sim))
info("ACCURACY: {:2.1%} RECALL: {:2.1%} SIMILARITY: {:.4} -> {:.1f}°".format(acc, rec, sim, angle))
     
Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
07:48:34 [INFO] TEST: ENCODING...

07:48:50 [INFO] EVALUATING...

07:48:56 [INFO] ACCURACY: 81.0% RECALL: 81.0% SIMILARITY: 0.6952 -> 46.0°

Results of our net

Before training, so with the default net Custom Clip not trained, the results are

14:29:32 [INFO] TEST: ENCODING...

14:29:33 [INFO] EVALUATING...

14:29:35 [INFO] ACCURACY: 8.0% RECALL: 8.0% SIMILARITY: 0.0747 -> 85.7°
Samples randomly chosen from the test set results in an 80% of accuracy and 0.64 of cosine similarity

14:05:59 [INFO] TEST: ENCODING...

14:06:00 [INFO] EVALUATING...

14:06:02 [INFO] ACCURACY: 79.0% RECALL: 79.0% SIMILARITY: 0.6436 -> 49.9°
Good improvement from the start point

It is possible to see how the accuracy score drastically improve after the training period, passing from 8% to 79%, which show how the fine-tuning process of the model return better results on linkng text to image on the RefCOCO Dataset, and being able to classify the text to retrieve the bounding boxes for the detctor model. The accuracy returned is then exactly the total recall of the model, that is cause by the multicalss classification problem, and in this case the accuracy matches the recall. The last important metric is regarding the semantic similarity, which is measured by using the cosine similarity between the tensors returned by the model. In this case it is immidiate the improvements of the similarity, which are similar to the accuracy's one, showing how the wieghts of the model have been updated to yield better results overall.

Final Program
Final program is only for the purpose to test the net. It allows you to load images (also custom if you want) and a text description (for the moment taken from dataset, but it seems it works even better if you take from personal images and personal input). So, simply try it.


def putTextBg(img, text, org, font, size, fg_color, thickness, linetype, bg_color):
    # A support function to inser text with a background in an image
    text_size, _ = cv2.getTextSize(text, font, size, thickness)
    text_w, text_h = text_size
    img = cv2.rectangle(img, (org[0]-2, org[1]-text_h-2), (org[0] + text_w + 2, org[1] + 5), bg_color, -1)
    img = cv2.putText (img, text, org, font, size, fg_color, thickness, linetype)
    return img
     

from google.colab.patches import cv2_imshow
def final_program(clip_model=None):
    annotations_file = 'dataset/refcocog/annotations/refs(umd).p'
    root_imgs = 'dataset/refcocog/images'

    # Load the model
    if clip_model is None:
        clip_model = CustomClip(device=get_device(), norm=False)
    _, clip_processor = clip_model.__get_model__()
    sample_size = 300
    info("Total size: "+str(sample_size))
    test_data = RefCOCO(annotations_file = annotations_file, img_dir=root_imgs, preprocess=clip_processor, split_type='test', device=get_device(), sample_size=sample_size)

    # Pick random images and description and draw a rectangle according to the bounding box
    for i in range(3):
        index = random.randint(0, test_data.__len__())
        _, image = test_data.__getimg__(index)
        textual_desc = random.choice(test_data.__gettext__(index))

        img = cv2.imread(image)
        item, prob = clip_model.__get_boxes__(image, textual_desc)
        if item is not None:

            cv2.rectangle(img, (item["xmin"], item["ymin"]), (item["xmax"], item["ymax"]), (0,127,0), 3)
            info("{:05d}: {} --> {}\n\033[92m{:2.1%}\033[0m".format(index, image, textual_desc, prob))
            img = putTextBg(img, textual_desc + " " + str(int(float(prob)*100))+"%", (0,10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA, (0,127,0))
            cv2_imshow(img)
            if cv2.waitKey(0) == 27: #if you press ESC button, you will exit the program
                return
            cv2.destroyAllWindows()

if __name__ == "__main__":
    model_path = 'model/FINAL_MODEL_personal_model_Adadelta_60K-16-aug-sigmoid.pt'
    clip_model = CustomClip(device=get_device(), norm=False)
    clip_model, epoch, loss = load_model(clip_model, model_path)
    final_program(clip_model)
    cv2.destroyAllWindows()
     
Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
13:21:47 [INFO] Total size: 300

13:21:49 [INFO] TEST: ENCODING...

13:21:52 [INFO] 00013: dataset/refcocog/images/COCO_train2014_000000553428.jpg --> A parked white Ford SUV
41.0%


13:21:53 [INFO] 00076: dataset/refcocog/images/COCO_train2014_000000075924.jpg --> Someone's hand brushing the teeth of the child
100.0%


13:21:53 [INFO] 00284: dataset/refcocog/images/COCO_train2014_000000044123.jpg --> the swan closest to photographer
51.1%


Conclusions

The goal of the project was to apply a visual grounding task to the RefCOCOg Dataset, which means linking the descriptions of the images to the right bounding boxes. As the Neural Network was returning positive outcomes during the development process, it also retrieved some issues that should need to be taken into account for future works. The main aspect was for sure the absence of training data with boxes already annotated, which made the final results not as precise and accurate as hoped, but still a good starting point for future improvements.

Strengths and Weaknesses

Strengths:

Fast training: after few epochs with lot of data converge with an optimum accuracy on validation and test set. Training is tested with a GPU Tesla V100: the time for each epoch is around 8 minutes
Tuned parameters: after some tests, parameters were tuned to get the best performance and result
Robust architecture allows to train better the weights: there are some other layers over the standard clip ones
Data augmentation for images to have even more data to train with and to get even better results in the training phase
Weaknesses:

Not so good performance in predicting bounding boxes: dataset RefCOCOg was without ground truth on boxes, so the training wasn't possible on them
What we learnt

From the quite 0-knowledge when our team started, we have learnt a lot of things:

We didn't know how to use the workspace PyTorch: now we have a deeper knowledge on how it works and the power functionalities that it offers
How to manage data? How to train a net? How to choose parameters? These are only few questions that we met during this course and this project: it's a good way to learn how to solve complex problems
Step by step, learning each time we work in this project something new and bringing the new knowledge to the upper "layers"
Read lot of papers / articles to tune the algorithm in the best way possible and reducing time in trying useless techniques or parameters
Improvements

To improve better the network for its primary goal, we would need a dataset with the bounding boxes where there are the objects that the descriptions refer to: an example could be LVIS. This would represents a very good improvements for the net, but it's not possible with RefCOCOg, since it has no data regarding where the objects are.

A possible future step then would be to use the current model to generate a first draft of annotations with boxes, then by refining and fine-tuning the model overwrite the less accurate boxes with the new once. Even though there are margin of improvments and different aspects to implement for future works, yet the model is capable to perform the visual grounding task